<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row = one page ,the page`s average performance from 2/4 to 30/6

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")
SRC = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"


START, END = "2026-04-01", "2026-05-31"

df = con.execute(f"""
SELECT content_hash_id,
       client_hash_id,
       SUM(gsc_impressions) * 1.0 / COUNT(*) AS avg_impressions
FROM read_parquet('{SRC}', hive_partitioning=true)
WHERE month IN ('2026-04', '2026-05')
  AND report_date BETWEEN DATE '{START}' AND DATE '{END}'
GROUP BY client_hash_id, content_hash_id
""").df()

print(df.shape)
df.head()

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(389153, 3)


,content_hash_id,client_hash_id,avg_impressions
0,content_76c1f31e2b38f054,client_62f4a7e64f5e0096,8.622951
1,content_ffc5ab4b34aab1f8,client_62f4a7e64f5e0096,17.622951
2,content_9739856fc83dc1ca,client_62f4a7e64f5e0096,19.360656
3,content_d47ba5533f9c8573,client_62f4a7e64f5e0096,0.000000
4,content_50266f97d6233542,client_62f4a7e64f5e0096,0.327869


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## Fields

**Features**

(report_date from 2026-04-02 to 2026-05-31)

From `fact_content_daily_performance` :
`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `ga4_pageviews`, `ga4_users`

From `dim_content`:
`content_created_date`, `keyword_hash_id`, `main_intent`, `backlinks`

From `fact_content_query_90d`:
`impressions_90d`, `impressions_last30`

**Label**

`gsc_impressions` from `fact_content_daily_performance` where `report_date` is in June 2026.

**Context**

`client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_start`, `ga4_data_start`, `is_published`, `is_deleted`, `last_optimized_date`

**Excluded**

All other columns. The `*_90d` columns of `fact_content_query_90d` cover June (the label period), so they would leak the label. The remaining columns are not expected to have an effect on the decline of a page.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:


REL = "hf://datasets/FlyRank/internship-warehouse"
SRC = f"{REL}/fact_content_daily_performance/**/*.parquet"

F_START, F_END = "2026-04-02", "2026-05-31"   # فترة الـ features
L_START, L_END = "2026-06-01", "2026-06-30"   # يونيو

# استعلام 1: نقرا الجدول مرة واحدة ونخزّن النتيجة في جدول raw
con.execute(f"""
CREATE OR REPLACE TABLE raw AS
WITH pages AS (
    SELECT c.client_hash_id, c.content_hash_id,
           CAST(cl.gsc_data_start AS DATE) AS gsc_start
    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN read_parquet('{REL}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.is_published = TRUE
      AND c.is_deleted = FALSE
      AND (c.last_optimized_date IS NULL
           OR c.last_optimized_date <  DATE '{L_START}'
           OR c.last_optimized_date >  DATE '{L_END}')
      AND cl.gsc_data_start IS NOT NULL
)
SELECT p.client_hash_id, p.content_hash_id,
       COUNT(DISTINCT f.report_date) FILTER (WHERE f.report_date BETWEEN DATE '{F_START}' AND DATE '{F_END}') AS n_days,
       COUNT(DISTINCT f.report_date) FILTER (WHERE f.report_date BETWEEN DATE '{L_START}' AND DATE '{L_END}') AS n_days_june,
       SUM(f.gsc_impressions)        FILTER (WHERE f.report_date BETWEEN DATE '{F_START}' AND DATE '{F_END}') AS impr_features
FROM read_parquet('{SRC}', hive_partitioning=true) f
JOIN pages p
  ON f.client_hash_id  = p.client_hash_id
 AND f.content_hash_id = p.content_hash_id
WHERE f.month IN ('2026-04', '2026-05', '2026-06')
  AND f.report_date BETWEEN DATE '{F_START}' AND DATE '{L_END}'
  AND f.report_date >= p.gsc_start
GROUP BY 1, 2
""")

# توزيع مجموع الظهور عشان تختار الحد الأدنى
print(con.execute("""
SELECT COUNT(*) AS n_pages,
       QUANTILE_CONT(impr_features, 0.10) AS p10,
       QUANTILE_CONT(impr_features, 0.25) AS p25,
       QUANTILE_CONT(impr_features, 0.50) AS median,
       QUANTILE_CONT(impr_features, 0.75) AS p75,
       QUANTILE_CONT(impr_features, 0.90) AS p90
FROM raw
""").df())

# استعلام 2: نطبّق الحد الأدنى (غيّر الرقم وأعد الخلية دي بس)
MIN_IMPR = 300
df = con.execute(f"""
SELECT client_hash_id, content_hash_id, n_days, n_days_june
FROM raw
WHERE impr_features >= {MIN_IMPR}
""").df()

print(f"\nبعد الحد الأدنى ({MIN_IMPR}):", len(df), "صفحة |", df.client_hash_id.nunique(), "عميل")
print(df[["n_days", "n_days_june"]].describe())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_pages  p10  p25  median    p75     p90
0   363927  0.0  0.0     8.0  241.0  1436.0

بعد الحد الأدنى (300): 79320 صفحة | 49 عميل
             n_days   n_days_june
count  79320.000000  79320.000000
mean      57.262494     29.898638
std        8.593697      0.625555
min        6.000000     25.000000
25%       60.000000     30.000000
50%       60.000000     30.000000
75%       60.000000     30.000000
max       60.000000     30.000000


,client_hash_id,content_hash_id,n_days,n_days_june
0,client_2094c6eb080311d5,content_15200126d986d645,60,30
1,client_2094c6eb080311d5,content_15caf3515edbfac3,53,30
2,client_2094c6eb080311d5,content_15f758d8bd5341c5,60,30
3,client_2094c6eb080311d5,content_16157539a54a6404,57,30
4,client_2094c6eb080311d5,content_1630bd7c0684d82c,60,30


In [4]:
df.head(20)

,client_hash_id,content_hash_id,n_days,n_days_june
0,client_2094c6eb080311d5,content_15200126d986d645,60,30
1,client_2094c6eb080311d5,content_15caf3515edbfac3,53,30
2,client_2094c6eb080311d5,content_15f758d8bd5341c5,60,30
3,client_2094c6eb080311d5,content_16157539a54a6404,57,30
4,client_2094c6eb080311d5,content_1630bd7c0684d82c,60,30
5,client_2094c6eb080311d5,content_16515c58d44f1f37,60,30
6,client_2094c6eb080311d5,content_167ea9e53c1a8e6e,60,30
7,client_2094c6eb080311d5,content_1696131c8c10f3cd,60,30
8,client_2094c6eb080311d5,content_16a0917ed541a052,57,30
9,client_2094c6eb080311d5,content_16dc713628278346,60,30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.